## Audio books analysis
This model will analyse the data of past clients to train and then be able to forecast whether a customer is likely to buy again or not.

# Gemini's take at preprocessing the Audiobook dataset

In [1]:
import numpy as np

# 1. Load and Strip ID
raw_csv_data = np.loadtxt('../../../../statistics/python/audiobooks/Audiobooks_data.csv', delimiter=',')
# Remove first column (ID) and separate Features (X) from Labels (y)
all_features = raw_csv_data[:, 1:-1]
all_labels = raw_csv_data[:, -1]

# 2. Identify and Count Classes
unique_classes, counts = np.unique(all_labels, return_counts=True)
minority_class_size = np.min(counts)

print(f"Classes found: {unique_classes}")
print(f"Balancing all classes to {minority_class_size} samples each.")

# 3. Balanced Index Selection (The Generalized Part)
balanced_indices = []

for c in unique_classes:
    # Get indices where the label matches class 'c'
    class_indices = np.where(all_labels == c)[0]
    
    # Shuffle these specific indices so we pick randomly
    np.random.shuffle(class_indices)
    
    # Take only as many as the minority class has
    balanced_indices.append(class_indices[:minority_class_size])

# Combine all selected indices into one flat array
balanced_indices = np.concatenate(balanced_indices)

# 4. Final Shuffle
# This mixes the classes together so they aren't grouped
np.random.shuffle(balanced_indices)

# 5. Apply to data
X_balanced = all_features[balanced_indices]
y_balanced = all_labels[balanced_indices]

Classes found: [0. 1.]
Balancing all classes to 2237 samples each.


### Split dataset in 80/10/10 for train/validate/test

In [2]:
# 1. Determine the total count of balanced samples
samples_count = balanced_indices.shape[0]

# 2. Calculate the split points (cumulative)
# 80% for training
train_samples_count = int(0.8 * samples_count)
# 10% for validation (added to the 80% to find the index)
validation_samples_count = int(0.1 * samples_count)

# 3. Slice the data using the shuffled balanced indices
# Training set (from 0 to 80%)
train_inputs = all_features[balanced_indices[:train_samples_count]]
train_targets = all_labels[balanced_indices[:train_samples_count]]

# Validation set (from 80% to 90%)
validation_inputs = all_features[balanced_indices[train_samples_count : train_samples_count + validation_samples_count]]
validation_targets = all_labels[balanced_indices[train_samples_count : train_samples_count + validation_samples_count]]

# Test set (the remaining 10%)
test_inputs = all_features[balanced_indices[train_samples_count + validation_samples_count:]]
test_targets = all_labels[balanced_indices[train_samples_count + validation_samples_count:]]

# 4. Verify the distribution (Optional but recommended)
print(f"Training: {train_inputs.shape[0]} samples")
print(f"Validation: {validation_inputs.shape[0]} samples")
print(f"Test: {test_inputs.shape[0]} samples")

# 5. Save as .npz for tomorrow's class
#np.savez('Data_train', inputs=train_inputs, targets=train_targets)
#np.savez('Data_validation', inputs=validation_inputs, targets=validation_targets)
#np.savez('Data_test', inputs=test_inputs, targets=test_targets)

Training: 3579 samples
Validation: 447 samples
Test: 448 samples


In [3]:
# Check that the '0'- and '1'-classes are balanced in each of the three datasets
print(np.sum(train_targets), train_inputs.shape[0], np.sum(train_targets)/train_inputs.shape[0])
print(np.sum(validation_targets), validation_inputs.shape[0], np.sum(validation_targets)/validation_inputs.shape[0])
print(np.sum(test_targets), test_inputs.shape[0], np.sum(test_targets)/test_inputs.shape[0])


1791.0 3579 0.5004191114836547
232.0 447 0.5190156599552572
214.0 448 0.47767857142857145


In [4]:
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('../../../../statistics/python/audiobooks/Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

### Machine learing part
Preprocesing is done. It can be reused for other problems.
From here on, we will start from the .npz files.